In [1]:
#easy
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)

qc.cx(0, 1)
qc.h(0)

qc.measure([0, 1], [0, 1])

simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()

print("Message sent: 00")
print("Decoded result:", result.get_counts())

Message sent: 00
Decoded result: {'00': 1024}


In [3]:
#medium
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
simulator = AerSimulator()
messages = ["00", "01", "10", "11"]
print("Superdense Coding Results")
print("-" * 35)
for message in messages:
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    if message == "01":
        qc.x(0)

    elif message == "10":
        qc.z(0)

    elif message == "11":
        qc.z(0)
        qc.x(0)
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1024).result()
    counts = result.get_counts()
    decoded = max(counts, key=counts.get)
    print("Message:", message, " -> Decoded:", decoded)

Superdense Coding Results
-----------------------------------
Message: 00  -> Decoded: 00
Message: 01  -> Decoded: 10
Message: 10  -> Decoded: 01
Message: 11  -> Decoded: 11


In [5]:
#hard
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
simulator = AerSimulator()
messages = ["00", "01", "10", "11"]
print("Superdense Coding with Bit-Flip Error")
print("-" * 50)
for message in messages:
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    if message == "01":
        qc.x(0)

    elif message == "10":
        qc.z(0)

    elif message == "11":
        qc.z(0)
        qc.x(0)
    qc.x(0)

    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1024).result()
    counts = result.get_counts()
    decoded = max(counts, key=counts.get)
    accuracy = counts.get(message, 0) / 1024 * 100
    print("Message:", message)
    print("Decoded:", decoded)
    print("Accuracy:", round(accuracy, 2), "%")
    print()

Superdense Coding with Bit-Flip Error
--------------------------------------------------
Message: 00
Decoded: 10
Accuracy: 0.0 %

Message: 01
Decoded: 00
Accuracy: 0.0 %

Message: 10
Decoded: 11
Accuracy: 0.0 %

Message: 11
Decoded: 01
Accuracy: 0.0 %



In [7]:
#real world
number_of_messages = 100
bits_per_message = 2
classical_bits = number_of_messages * bits_per_message
quantum_qubits = number_of_messages
print("Superdense Coding Bandwidth Comparison")
print("-" * 45)

print("Number of messages:", number_of_messages)
print("Bits per message:", bits_per_message)

print("\nClassical communication:")
print("Classical bits transmitted:", classical_bits)

print("\nSuperdense coding:")
print("Physical qubits transmitted:", quantum_qubits)
print("Classical information encoded:", classical_bits, "bits")

print("\nResult:")
print("Superdense coding transmits 2 classical bits")
print("through one transmitted qubit, assuming")
print("a shared entangled pair is already available.")

Superdense Coding Bandwidth Comparison
---------------------------------------------
Number of messages: 100
Bits per message: 2

Classical communication:
Classical bits transmitted: 200

Superdense coding:
Physical qubits transmitted: 100
Classical information encoded: 200 bits

Result:
Superdense coding transmits 2 classical bits
through one transmitted qubit, assuming
a shared entangled pair is already available.


In [9]:
#challenge
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

fidelity = 0.90
shots = 2000
error_probability = 1 - fidelity
noise_model = NoiseModel()
single_qubit_error = depolarizing_error(
    error_probability, 1
)

two_qubit_error = depolarizing_error(
    error_probability, 2
)

noise_model.add_all_qubit_quantum_error(
    single_qubit_error,
    ["h", "x", "z"]
)

noise_model.add_all_qubit_quantum_error(
    two_qubit_error,
    ["cx"]
)

simulator = AerSimulator(noise_model=noise_model)
messages = ["00", "01", "10", "11"]
total_correct = 0
total_shots = 0

print("Noisy Superdense Coding Simulation")
print("-----------------------------------")
print("Specified entangled-pair fidelity:", fidelity)
print()
for message in messages:
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    if message == "01":
        qc.x(0)
    elif message == "10":
        qc.z(0)
    elif message == "11":
        qc.z(0)
        qc.x(0)
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    compiled = transpile(qc, simulator)
    result = simulator.run(
        compiled,
        shots=shots
    ).result()
    counts = result.get_counts()
    correct = counts.get(message, 0)
    accuracy = correct / shots * 100
    error_rate = 100 - accuracy
    total_correct += correct
    total_shots += shots
    print("Message:", message)
    print("Counts:", counts)
    print("Accuracy:", round(accuracy, 2), "%")
    print("Error rate:", round(error_rate, 2), "%")
    print()
overall_accuracy = total_correct / total_shots * 100
overall_error = 100 - overall_accuracy
print("-----------------------------------")
print("Overall Accuracy:", round(overall_accuracy, 2), "%")
print("Overall Error Rate:", round(overall_error, 2), "%")

Noisy Superdense Coding Simulation
-----------------------------------
Specified entangled-pair fidelity: 0.9

Message: 00
Counts: {'00': 2000}
Accuracy: 100.0 %
Error rate: 0.0 %

Message: 01
Counts: {'10': 1886, '00': 114}
Accuracy: 0.0 %
Error rate: 100.0 %

Message: 10
Counts: {'01': 1897, '00': 103}
Accuracy: 0.0 %
Error rate: 100.0 %

Message: 11
Counts: {'11': 1788, '10': 104, '01': 101, '00': 7}
Accuracy: 89.4 %
Error rate: 10.6 %

-----------------------------------
Overall Accuracy: 47.35 %
Overall Error Rate: 52.65 %
